# Fine-tune AgyssyyGPT on a free Colab GPU

This notebook runs **QLoRA** fine-tuning on your own `agys --log` conversations and
pushes the adapter to the Hugging Face Hub.

### Before you start
1. **Runtime → Change runtime type → T4 GPU.**
2. Add your Hugging Face **write** token
   (https://huggingface.co/settings/tokens) to **Secrets** (the key icon on the left),
   name it `HF_TOKEN`, and enable notebook access. If you skip that, the login cell
   will prompt you to paste it.
3. Have your data ready: use a Hub dataset id (e.g. `YOUR-NAME/agysyygpt-chats`), or
   upload `dataset.jsonl` with the files panel and set `DATASET` to its path.

> Free Colab GPUs are session-limited (a few hours). A 3B model on a few thousand
> examples trains in well under an hour.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/agyssyy/agysyygpt.git
%cd agysyygpt
!pip install -q -r training/requirements.txt

In [ ]:
import os

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from getpass import getpass

    os.environ.setdefault("HF_TOKEN", getpass("Hugging Face write token: "))

from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])

## Configure the run

Set `DATASET` to a Hub dataset id or an uploaded `dataset.jsonl` path, and `HF_REPO`
to where the trained adapter should be published.

In [ ]:
DATASET = "your-name/agysyygpt-chats"       # or "/content/dataset.jsonl"
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT = "outputs/agysyygpt-lora"
HF_REPO = "your-name/agysyygpt-lora"         # where to push the adapter
EPOCHS = 3
MAX_SEQ_LEN = 2048

In [ ]:
!python training/finetune_lora.py \
  --dataset {DATASET} \
  --base-model {BASE_MODEL} \
  --output {OUTPUT} \
  --epochs {EPOCHS} \
  --max-seq-len {MAX_SEQ_LEN} \
  --push-to-hub {HF_REPO}

## Next: run it locally with Ollama

Merge the adapter and export to GGUF (see `training/README.md`, step 4), then:

```sh
agys -p ollama -m agysyygpt
```